# **Cell 1: Imports, Configuration & Environment Setup**

In [ ]:
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
np.random.seed(42)
# กำหนด Path ตรงตาม Kaggle Environment ของคุณ
KAGGLE_DIR = '/kaggle/input/competitions/ml69-c3'
if os.path.exists(KAGGLE_DIR):
    train_path = os.path.join(KAGGLE_DIR, 'train.csv')
    test_path = os.path.join(KAGGLE_DIR, 'test.csv')
    sub_sample_path = os.path.join(KAGGLE_DIR, 'sample_submission.csv')
    OUTPUT_DIR = '/kaggle/working'
elif os.path.exists('/kaggle/input'):
    # สำรองกรณี path โฟลเดอร์มีการเปลี่ยน
    train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
    test_path = glob.glob('/kaggle/input/**/test.csv', recursive=True)[0]
    sub_sample_path = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)[0]
    OUTPUT_DIR = '/kaggle/working'
else:
    # รันบนเครื่องตนเอง (Local)
    train_path = './train.csv'
    test_path = './test.csv'
    sub_sample_path = './sample_submission.csv'
    OUTPUT_DIR = '.'
submission_path = os.path.join(OUTPUT_DIR, 'submission.csv')
print("="*65)
print(f"📥 Train Path:       {train_path}")
print(f"📥 Test Path:        {test_path}")
print(f"📥 Sample Sub Path:  {sub_sample_path}")
print(f"📤 Output Save Path: {submission_path}")
print("="*65)
assert os.path.exists(train_path), f"Cannot find {train_path}"
assert os.path.exists(test_path), f"Cannot find {test_path}"
print("All files verified successfully!")

# **Cell 2: Advanced Biomechanical Feature Engineering Engine**

In [ ]:
def extract_channel_stats(sig, prefix):
    """
    Extract comprehensive time-domain, frequency-domain (FFT),
    cadence (autocorrelation), and shape features from a 2D signal array (N_samples, 64).
    """
    feats = {}
    N = sig.shape[1]
    
    # --- 1. Basic Statistical Moments ---
    m = np.mean(sig, axis=1)
    s = np.std(sig, axis=1)
    mx = np.max(sig, axis=1)
    mn = np.min(sig, axis=1)
    ptp = mx - mn
    
    feats[f'{prefix}_mean'] = m
    feats[f'{prefix}_std'] = s
    feats[f'{prefix}_max'] = mx
    feats[f'{prefix}_min'] = mn
    feats[f'{prefix}_ptp'] = ptp
    
    q10 = np.percentile(sig, 10, axis=1)
    q25 = np.percentile(sig, 25, axis=1)
    q50 = np.median(sig, axis=1)
    q75 = np.percentile(sig, 75, axis=1)
    q90 = np.percentile(sig, 90, axis=1)
    
    feats[f'{prefix}_q10'] = q10
    feats[f'{prefix}_q25'] = q25
    feats[f'{prefix}_q50'] = q50
    feats[f'{prefix}_q75'] = q75
    feats[f'{prefix}_q90'] = q90
    feats[f'{prefix}_iqr'] = q75 - q25
    
    feats[f'{prefix}_rms'] = np.sqrt(np.mean(sig**2, axis=1))
    feats[f'{prefix}_mad'] = np.mean(np.abs(sig - m[:, None]), axis=1)
    feats[f'{prefix}_skew'] = stats.skew(sig, axis=1)
    feats[f'{prefix}_kurt'] = stats.kurtosis(sig, axis=1)
    
    # --- 2. Mean Crossing Rate ---
    centered = sig - m[:, None]
    crossings = np.sum((centered[:, :-1] * centered[:, 1:]) < 0, axis=1) / (N - 1)
    feats[f'{prefix}_mcr'] = crossings
    
    # --- 3. Autocorrelation & Cadence Detection ---
    padded = np.pad(centered, ((0, 0), (0, N)), mode='constant')
    f_sig = np.fft.fft(padded, axis=1)
    autocorr = np.fft.ifft(f_sig * np.conj(f_sig), axis=1).real[:, :N]
    autocorr = autocorr / (autocorr[:, 0:1] + 1e-8)
    
    feats[f'{prefix}_autocorr_lag1'] = autocorr[:, 1]
    feats[f'{prefix}_autocorr_lag2'] = autocorr[:, 2]
    feats[f'{prefix}_autocorr_lag4'] = autocorr[:, 4]
    
    # Search for fundamental step cadence in lags 6 to 30 (0.83 Hz to 4.16 Hz at 25 Hz)
    sub_ac = autocorr[:, 6:30]
    peak_lag = np.argmax(sub_ac, axis=1) + 6
    peak_val = np.max(sub_ac, axis=1)
    feats[f'{prefix}_cadence_freq'] = 25.0 / peak_lag
    feats[f'{prefix}_cadence_regularity'] = peak_val
    
    # --- 4. Frequency Domain Features (FFT) ---
    fft_vals = np.abs(np.fft.rfft(centered, axis=1))[:, 1:] # exclude DC
    freqs = np.fft.rfftfreq(N, d=1.0/25.0)[1:]
    
    fft_power = fft_vals**2
    total_power = np.sum(fft_power, axis=1) + 1e-8
    feats[f'{prefix}_fft_energy'] = total_power
    
    p = fft_power / total_power[:, None] + 1e-12
    feats[f'{prefix}_spec_entropy'] = -np.sum(p * np.log(p), axis=1)
    
    dom_idx = np.argmax(fft_vals, axis=1)
    feats[f'{prefix}_dom_freq'] = freqs[dom_idx]
    feats[f'{prefix}_dom_amp'] = np.max(fft_vals, axis=1)
    
    sum_fft = np.sum(fft_vals, axis=1) + 1e-8
    centroid = np.sum(fft_vals * freqs, axis=1) / sum_fft
    feats[f'{prefix}_spec_centroid'] = centroid
    spread = np.sqrt(np.sum(fft_vals * (freqs - centroid[:, None])**2, axis=1) / sum_fft)
    feats[f'{prefix}_spec_spread'] = spread
    
    # Biomechanical Frequency Bands
    b1 = (freqs >= 0.3) & (freqs < 1.2)  # Low posture sway
    b2 = (freqs >= 1.2) & (freqs < 2.2)  # Normal walking cadence
    b3 = (freqs >= 2.2) & (freqs < 3.5)  # Running cadence
    b4 = (freqs >= 3.5) & (freqs < 6.0)  # Impact harmonics
    b5 = (freqs >= 6.0)                  # High impact shocks
    
    e1 = np.sum(fft_power[:, b1], axis=1)
    e2 = np.sum(fft_power[:, b2], axis=1)
    e3 = np.sum(fft_power[:, b3], axis=1)
    e4 = np.sum(fft_power[:, b4], axis=1)
    e5 = np.sum(fft_power[:, b5], axis=1)
    
    feats[f'{prefix}_band_03_12'] = e1
    feats[f'{prefix}_band_12_22'] = e2
    feats[f'{prefix}_band_22_35'] = e3
    feats[f'{prefix}_band_35_60'] = e4
    feats[f'{prefix}_band_60_12'] = e5
    
    feats[f'{prefix}_ratio_walk_run'] = e2 / (e3 + 1e-6)
    feats[f'{prefix}_ratio_low_high'] = (e1 + e2) / (e3 + e4 + e5 + 1e-6)
    
    return feats

In [ ]:
def extract_channel_stats(sig, prefix):
    """
    Vectorized statistical & frequency-domain feature extraction across 64-point signals.
    sig: (N, 64) numpy array
    """
    feats = {}
    N = sig.shape[1]
    
    # 1. Summary Moments
    m = np.mean(sig, axis=1)
    s = np.std(sig, axis=1)
    mx = np.max(sig, axis=1)
    mn = np.min(sig, axis=1)
    
    feats[f'{prefix}_mean'] = m
    feats[f'{prefix}_std'] = s
    feats[f'{prefix}_max'] = mx
    feats[f'{prefix}_min'] = mn
    feats[f'{prefix}_ptp'] = mx - mn
    
    q10 = np.percentile(sig, 10, axis=1)
    q25 = np.percentile(sig, 25, axis=1)
    q50 = np.median(sig, axis=1)
    q75 = np.percentile(sig, 75, axis=1)
    q90 = np.percentile(sig, 90, axis=1)
    
    feats[f'{prefix}_q10'] = q10
    feats[f'{prefix}_q25'] = q25
    feats[f'{prefix}_q50'] = q50
    feats[f'{prefix}_q75'] = q75
    feats[f'{prefix}_q90'] = q90
    feats[f'{prefix}_iqr'] = q75 - q25
    
    feats[f'{prefix}_rms'] = np.sqrt(np.mean(sig**2, axis=1))
    feats[f'{prefix}_mad'] = np.mean(np.abs(sig - m[:, None]), axis=1)
    feats[f'{prefix}_skew'] = stats.skew(sig, axis=1)
    feats[f'{prefix}_kurt'] = stats.kurtosis(sig, axis=1)
    
    # 2. Mean Crossing Rate
    centered = sig - m[:, None]
    crossings = np.sum((centered[:, :-1] * centered[:, 1:]) < 0, axis=1) / (N - 1)
    feats[f'{prefix}_mcr'] = crossings
    
    # 3. Autocorrelation & Step Cadence Detection via FFT
    padded = np.pad(centered, ((0, 0), (0, N)), mode='constant')
    f_sig = np.fft.fft(padded, axis=1)
    autocorr = np.fft.ifft(f_sig * np.conj(f_sig), axis=1).real[:, :N]
    autocorr = autocorr / (autocorr[:, 0:1] + 1e-8)
    
    feats[f'{prefix}_autocorr_lag1'] = autocorr[:, 1]
    feats[f'{prefix}_autocorr_lag2'] = autocorr[:, 2]
    feats[f'{prefix}_autocorr_lag4'] = autocorr[:, 4]
    
    # Search for fundamental cadence in lags 6 to 30 (0.83 Hz to 4.16 Hz at 25 Hz)
    sub_ac = autocorr[:, 6:30]
    peak_lag = np.argmax(sub_ac, axis=1) + 6
    peak_val = np.max(sub_ac, axis=1)
    feats[f'{prefix}_cadence_freq'] = 25.0 / peak_lag
    feats[f'{prefix}_cadence_regularity'] = peak_val
    
    # 4. Spectral Analysis (FFT)
    fft_vals = np.abs(np.fft.rfft(centered, axis=1))[:, 1:]
    freqs = np.fft.rfftfreq(N, d=1.0/25.0)[1:]
    
    fft_power = fft_vals**2
    total_power = np.sum(fft_power, axis=1) + 1e-8
    feats[f'{prefix}_fft_energy'] = total_power
    
    p = fft_power / total_power[:, None] + 1e-12
    feats[f'{prefix}_spec_entropy'] = -np.sum(p * np.log(p), axis=1)
    
    dom_idx = np.argmax(fft_vals, axis=1)
    feats[f'{prefix}_dom_freq'] = freqs[dom_idx]
    feats[f'{prefix}_dom_amp'] = np.max(fft_vals, axis=1)
    
    sum_fft = np.sum(fft_vals, axis=1) + 1e-8
    centroid = np.sum(fft_vals * freqs, axis=1) / sum_fft
    feats[f'{prefix}_spec_centroid'] = centroid
    spread = np.sqrt(np.sum(fft_vals * (freqs - centroid[:, None])**2, axis=1) / sum_fft)
    feats[f'{prefix}_spec_spread'] = spread
    
    b1 = (freqs >= 0.3) & (freqs < 1.2)  # Posture sway
    b2 = (freqs >= 1.2) & (freqs < 2.2)  # Walking cadence
    b3 = (freqs >= 2.2) & (freqs < 3.5)  # Running cadence
    b4 = (freqs >= 3.5) & (freqs < 6.0)  # Gait harmonics
    b5 = (freqs >= 6.0)                  # Landing impact shocks
    
    e1 = np.sum(fft_power[:, b1], axis=1)
    e2 = np.sum(fft_power[:, b2], axis=1)
    e3 = np.sum(fft_power[:, b3], axis=1)
    e4 = np.sum(fft_power[:, b4], axis=1)
    e5 = np.sum(fft_power[:, b5], axis=1)
    
    feats[f'{prefix}_band_03_12'] = e1
    feats[f'{prefix}_band_12_22'] = e2
    feats[f'{prefix}_band_22_35'] = e3
    feats[f'{prefix}_band_35_60'] = e4
    feats[f'{prefix}_band_60_12'] = e5
    
    feats[f'{prefix}_ratio_walk_run'] = e2 / (e3 + 1e-6)
    feats[f'{prefix}_ratio_low_high'] = (e1 + e2) / (e3 + e4 + e5 + 1e-6)
    
    return feats

In [ ]:
def compute_all_features_fast(df, subj_gravity_dict=None):
    """
    Computes Orientation-Invariant Biomechanical Features in pure vectorized NumPy.
    Runs in ~3-4 seconds.
    """
    N_samples = len(df)
    ax_cols = [f'ax_{i}' for i in range(64)]
    ay_cols = [f'ay_{i}' for i in range(64)]
    az_cols = [f'az_{i}' for i in range(64)]
    
    X_ax = df[ax_cols].values
    X_ay = df[ay_cols].values
    X_az = df[az_cols].values
    subjects = df['subject_id'].values
    
    # 1. Subject Gravity Vector (Orientation Reference)
    if subj_gravity_dict is None:
        subj_gravity_dict = {}
        for s, sub_df in df.groupby('subject_id'):
            mx = sub_df[ax_cols].values.mean()
            my = sub_df[ay_cols].values.mean()
            mz = sub_df[az_cols].values.mean()
            norm = np.sqrt(mx**2 + my**2 + mz**2) + 1e-8
            subj_gravity_dict[s] = (mx / norm, my / norm, mz / norm)
            
    s_gx = np.array([subj_gravity_dict[s][0] for s in subjects])[:, None]
    s_gy = np.array([subj_gravity_dict[s][1] for s in subjects])[:, None]
    s_gz = np.array([subj_gravity_dict[s][2] for s in subjects])[:, None]
    
    # 2. Window Gravity Vector
    w_gx = X_ax.mean(axis=1, keepdims=True)
    w_gy = X_ay.mean(axis=1, keepdims=True)
    w_gz = X_az.mean(axis=1, keepdims=True)
    w_gn = np.sqrt(w_gx**2 + w_gy**2 + w_gz**2) + 1e-8
    wgx, wgy, wgz = w_gx / w_gn, w_gy / w_gn, w_gz / w_gn
    
    # 3. Coordinate Projections (Vertical & Horizontal Planes)
    vm = np.sqrt(X_ax**2 + X_ay**2 + X_az**2)
    av_s = X_ax * s_gx + X_ay * s_gy + X_az * s_gz
    h_vec_x = X_ax - av_s * s_gx
    h_vec_y = X_ay - av_s * s_gy
    h_vec_z = X_az - av_s * s_gz
    ah_s = np.sqrt(np.maximum(0, h_vec_x**2 + h_vec_y**2 + h_vec_z**2))
    
    av_w = X_ax * wgx + X_ay * wgy + X_az * wgz
    ah_w = np.sqrt(np.maximum(0, vm**2 - av_w**2))
    
    # 4. Dynamics & Jerks
    dyn_x = X_ax - w_gx
    dyn_y = X_ay - w_gy
    dyn_z = X_az - w_gz
    dyn_mag = np.sqrt(dyn_x**2 + dyn_y**2 + dyn_z**2)
    
    jx = np.diff(X_ax, axis=1)
    jy = np.diff(X_ay, axis=1)
    jz = np.diff(X_az, axis=1)
    j3d = np.sqrt(jx**2 + jy**2 + jz**2)
    
    jvm = np.diff(vm, axis=1)
    jav_s = np.diff(av_s, axis=1)
    jah_s = np.diff(ah_s, axis=1)
    
    # 5. Batch SVD: Forward vs Lateral Motion
    h_stack = np.stack([h_vec_x, h_vec_y, h_vec_z], axis=1)
    h_cent = h_stack - np.mean(h_stack, axis=2, keepdims=True)
    cov_h_batch = np.matmul(h_cent, h_cent.transpose(0, 2, 1)) / 63.0
    _, eigvecs_h = np.linalg.eigh(cov_h_batch)
    v_forward = eigvecs_h[:, :, 2]
    v_lateral = eigvecs_h[:, :, 1]
    af_s = np.sum(h_stack * v_forward[:, :, None], axis=1)
    al_s = np.sum(h_stack * v_lateral[:, :, None], axis=1)
    
    # 6. Vertical Dynamic Acceleration & Velocity Proxy
    av_dyn = av_s - np.mean(av_s, axis=1, keepdims=True)
    v_vertical = np.cumsum(av_dyn, axis=1) * (1.0 / 25.0)
    
    # Extract Base Channels
    all_feats = {}
    channels = [
        (vm, 'vm'),
        (av_s, 'av_s'),
        (ah_s, 'ah_s'),
        (av_w, 'av_w'),
        (ah_w, 'ah_w'),
        (dyn_mag, 'dyn_mag'),
        (af_s, 'af_s'),
        (al_s, 'al_s'),
        (v_vertical, 'vv'),
        (j3d, 'j3d'),
        (jvm, 'jvm'),
        (jav_s, 'jav_s'),
        (jah_s, 'jah_s')
    ]
    
    for sig, pfx in channels:
        all_feats.update(extract_channel_stats(sig, pfx))
        
    # 7. Specialized Features for Walk vs Stairs_Up vs Stairs_Down
    # (a) Vertical Mechanical Power = a_V * v_V
    power_v = av_dyn * v_vertical
    all_feats['power_v_mean'] = np.mean(power_v, axis=1)
    all_feats['power_v_std'] = np.std(power_v, axis=1)
    all_feats['power_v_pos_ratio'] = np.mean(power_v > 0, axis=1)
    all_feats['power_v_pos_work'] = np.sum(np.maximum(0, power_v), axis=1)
    all_feats['power_v_neg_work'] = np.sum(np.maximum(0, -power_v), axis=1)
    all_feats['power_v_work_ratio'] = all_feats['power_v_pos_work'] / (all_feats['power_v_neg_work'] + 1e-6)
    
    # (b) Vertical Impulse Ratio (Positive lifting thrust vs Negative landing impact)
    pos_impulse = np.sum(np.maximum(0, av_dyn), axis=1)
    neg_impulse = np.sum(np.maximum(0, -av_dyn), axis=1)
    all_feats['impulse_v_ratio'] = pos_impulse / (neg_impulse + 1e-6)
    
    # (c) Impact Steepness (Landing shock of stairs down)
    all_feats['peak_fall_slope'] = np.max(np.abs(np.diff(av_s, axis=1)), axis=1)
    all_feats['jerk_v_asymmetry'] = np.max(jav_s, axis=1) / (np.abs(np.min(jav_s, axis=1)) + 1e-6)
    all_feats['acc_v_asymmetry'] = (np.max(av_s, axis=1) - np.mean(av_s, axis=1)) / (np.abs(np.min(av_s, axis=1) - np.mean(av_s, axis=1)) + 1e-6)
    all_feats['duty_cycle_pos_v'] = np.mean(av_dyn > 0, axis=1)
    
    # (d) Harmonic Gait Symmetry (Bilateral Step Symmetry)
    # Walking on flat ground is bilaterally symmetric (even harmonics dominate)
    fft_pw = np.abs(np.fft.rfft(av_dyn, axis=1))[:, 1:16]**2
    even_h = np.sum(fft_pw[:, 1::2], axis=1)
    odd_h = np.sum(fft_pw[:, 0::2], axis=1)
    all_feats['gait_harmonic_symmetry'] = even_h / (odd_h + 1e-6)
    
    # (e) Forward vs Vertical Energy Ratio (High in walking, lower in stairs)
    all_feats['ratio_energy_f_v'] = all_feats['af_s_std'] / (all_feats['av_s_std'] + 1e-6)
    all_feats['ratio_energy_l_f'] = all_feats['al_s_std'] / (all_feats['af_s_std'] + 1e-6)
    all_feats['ratio_energy_v_vm'] = all_feats['av_s_std'] / (all_feats['vm_std'] + 1e-6)
    all_feats['ratio_energy_h_vm'] = all_feats['ah_s_std'] / (all_feats['vm_std'] + 1e-6)
    
    # 8. Vectorized Batch 3D Covariance Eigenvalues
    dyn_stack = np.stack([dyn_x, dyn_y, dyn_z], axis=1)
    cov_3d_batch = np.matmul(dyn_stack, dyn_stack.transpose(0, 2, 1)) / 63.0
    eigvals = np.linalg.eigvalsh(cov_3d_batch)
    
    all_feats['eig_0'] = eigvals[:, 2] # largest
    all_feats['eig_1'] = eigvals[:, 1] # middle
    all_feats['eig_2'] = eigvals[:, 0] # smallest
    trace = np.sum(eigvals, axis=1) + 1e-8
    all_feats['eig_trace'] = trace
    all_feats['eig_ratio_0'] = eigvals[:, 2] / trace
    all_feats['eig_ratio_1'] = eigvals[:, 1] / trace
    all_feats['eig_ratio_2'] = eigvals[:, 0] / trace
    all_feats['eig_linearity'] = (eigvals[:, 2] - eigvals[:, 1]) / (eigvals[:, 2] + 1e-8)
    all_feats['eig_planarity'] = (eigvals[:, 1] - eigvals[:, 0]) / (eigvals[:, 2] + 1e-8)
    all_feats['eig_sphericity'] = eigvals[:, 0] / (eigvals[:, 2] + 1e-8)
    
    feat_df = pd.DataFrame(all_feats).fillna(0)
    return feat_df, subj_gravity_dict

# **Cell 3: Load Data & Extract Features**

In [ ]:
print('Loading raw datasets...')
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

labels = sorted(train_df['activity'].unique())
label_map = {lbl: i for i, lbl in enumerate(labels)}
inv_label_map = {i: lbl for lbl, i in label_map.items()}
y = np.array([label_map[lbl] for lbl in train_df['activity'].values])
groups = train_df['subject_id'].values

print(f'Train samples: {len(train_df)} | Test samples: {len(test_df)}')
print(f'Classes ({len(labels)}): {labels}')

print('\nExtracting features from train and test...')
t0 = time.time()
X_train_feat, subj_g_dict = compute_all_features_fast(train_df)
X_test_feat, _ = compute_all_features_fast(test_df)
print(f'Feature extraction completed in {time.time()-t0:.2f}s! Total engineered features: {X_train_feat.shape[1]}')

# **Cell 4: 5-Fold GroupKFold Cross Validation & Multi-Model Training**

In [ ]:
N_SPLITS = 10
gkf = GroupKFold(n_splits=N_SPLITS)

oof_lgb = np.zeros((len(train_df), len(labels)))
oof_xgb = np.zeros((len(train_df), len(labels)))
oof_cat = np.zeros((len(train_df), len(labels)))

test_preds_lgb = np.zeros((len(test_df), len(labels)))
test_preds_xgb = np.zeros((len(test_df), len(labels)))
test_preds_cat = np.zeros((len(test_df), len(labels)))

print(f'\n>>> STAGE 1: Training {N_SPLITS}-Fold Ensemble <<<')
t_start = time.time()

for fold, (trn_idx, val_idx) in enumerate(gkf.split(X_train_feat, y, groups=groups)):
    X_tr, y_tr = X_train_feat.iloc[trn_idx], y[trn_idx]
    X_va, y_va = X_train_feat.iloc[val_idx], y[val_idx]
    
    # 1. LightGBM
    clf_lgb = lgb.LGBMClassifier(
        n_estimators=450,
        learning_rate=0.04,
        num_leaves=38,
        subsample=0.85,
        colsample_bytree=0.75,
        random_state=42 + fold,
        n_jobs=-1,
        verbose=-1
    )
    clf_lgb.fit(X_tr, y_tr)
    oof_lgb[val_idx] = clf_lgb.predict_proba(X_va)
    test_preds_lgb += clf_lgb.predict_proba(X_test_feat) / N_SPLITS
    
    # 2. XGBoost (Fast Histogram)
    clf_xgb = XGBClassifier(
        n_estimators=450,
        learning_rate=0.04,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.75,
        tree_method='hist',
        random_state=42 + fold,
        n_jobs=-1,
        verbosity=0,
        eval_metric='mlogloss'
    )
    clf_xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = clf_xgb.predict_proba(X_va)
    test_preds_xgb += clf_xgb.predict_proba(X_test_feat) / N_SPLITS

    # 3. CatBoost
    clf_cat = CatBoostClassifier(
        iterations=350,
        learning_rate=0.07,
        depth=5,
        random_seed=42 + fold,
        verbose=0,
        thread_count=-1
    )
    clf_cat.fit(X_tr, y_tr)
    oof_cat[val_idx] = clf_cat.predict_proba(X_va)
    test_preds_cat += clf_cat.predict_proba(X_test_feat) / N_SPLITS

print(f'Stage 1 Cross-Validation completed in {time.time()-t_start:.1f}s!')

stage1_blend_test = 0.40 * test_preds_lgb + 0.35 * test_preds_xgb + 0.25 * test_preds_cat

# STAGE 2: Iterative Pseudo-Labeling (Adapts model directly to the 12 Test Subjects)
print('\n>>> STAGE 2: Iterative Pseudo-Labeling Domain Adaptation <<<')
CONFIDENCE_THRESHOLD = 0.94
test_max_prob = np.max(stage1_blend_test, axis=1)
pseudo_mask = test_max_prob >= CONFIDENCE_THRESHOLD
pseudo_labels = np.argmax(stage1_blend_test, axis=1)

print(f'Found {np.sum(pseudo_mask)} high-confidence test samples (>={CONFIDENCE_THRESHOLD*100:.0f}%) for Pseudo-Labeling!')

if np.sum(pseudo_mask) > 100:
    X_pseudo = X_test_feat[pseudo_mask]
    y_pseudo = pseudo_labels[pseudo_mask]
    groups_pseudo = test_df['subject_id'].values[pseudo_mask]
    
    X_train_aug = pd.concat([X_train_feat, X_pseudo], ignore_index=True)
    y_aug = np.concatenate([y, y_pseudo])
    groups_aug = np.concatenate([groups, groups_pseudo])
    
    print(f'Training Final Stage 2 Ensemble on Augmented Dataset ({len(X_train_aug)} samples)...')
    test_preds_stage2 = np.zeros((len(test_df), len(labels)))
    
    gkf_aug = GroupKFold(n_splits=5)
    for fold, (trn_idx, _) in enumerate(gkf_aug.split(X_train_aug, y_aug, groups=groups_aug)):
        X_tr_aug, y_tr_aug = X_train_aug.iloc[trn_idx], y_aug[trn_idx]
        
        clf_lgb_final = lgb.LGBMClassifier(
            n_estimators=450,
            learning_rate=0.04,
            num_leaves=38,
            subsample=0.85,
            colsample_bytree=0.75,
            random_state=100 + fold,
            n_jobs=-1,
            verbose=-1
        )
        clf_lgb_final.fit(X_tr_aug, y_tr_aug)
        test_preds_stage2 += (0.50 * clf_lgb_final.predict_proba(X_test_feat)) / 5.0
        
        clf_xgb_final = XGBClassifier(
            n_estimators=450,
            learning_rate=0.04,
            max_depth=6,
            subsample=0.85,
            colsample_bytree=0.75,
            tree_method='hist',
            random_state=100 + fold,
            n_jobs=-1,
            verbosity=0,
            eval_metric='mlogloss'
        )
        clf_xgb_final.fit(X_tr_aug, y_tr_aug)
        test_preds_stage2 += (0.50 * clf_xgb_final.predict_proba(X_test_feat)) / 5.0
        
    final_test_prob = 0.50 * stage1_blend_test + 0.50 * test_preds_stage2
else:
    final_test_prob = stage1_blend_test

# **Cell 5: Ensemble Blending & Macro-F1 Threshold Multiplier Optimization**

In [ ]:
print('\n================ MODEL EVALUATION SUMMARY ================')
print(f'LightGBM OOF Macro-F1: {f1_score(y, np.argmax(oof_lgb, axis=1), average="macro"):.4f}')
print(f'XGBoost  OOF Macro-F1: {f1_score(y, np.argmax(oof_xgb, axis=1), average="macro"):.4f}')
print(f'CatBoost OOF Macro-F1: {f1_score(y, np.argmax(oof_cat, axis=1), average="macro"):.4f}')

oof_blend = 0.40 * oof_lgb + 0.35 * oof_xgb + 0.25 * oof_cat
raw_f1 = f1_score(y, np.argmax(oof_blend, axis=1), average="macro")
print(f'Stage 1 Ensemble OOF Macro-F1 (default argmax): {raw_f1:.4f}')

def optimize_multipliers(probs, y_true, n_classes=6):
    def loss(weights):
        scaled_p = probs * weights
        preds = np.argmax(scaled_p, axis=1)
        return -f1_score(y_true, preds, average='macro')
    
    init_w = np.ones(n_classes)
    res = minimize(loss, init_w, method='Powell', options={'maxiter': 50, 'ftol': 1e-4})
    best_weights = res.x / np.mean(res.x)
    return best_weights

print('\nOptimizing class probability multipliers for Macro-F1...')
best_multipliers = optimize_multipliers(oof_blend, y, n_classes=len(labels))
for i, lbl in enumerate(labels):
    print(f'Class {lbl:12s} multiplier: {best_multipliers[i]:.4f}')

calibrated_oof_preds = np.argmax(oof_blend * best_multipliers, axis=1)
final_oof_f1 = f1_score(y, calibrated_oof_preds, average='macro')
print(f'\n>>> FINAL CALIBRATED ENSEMBLE OOF MACRO-F1: {final_oof_f1:.4f} <<<')

print('\nClassification Report:')
print(classification_report(y, calibrated_oof_preds, target_names=labels, digits=4))

print('\nConfusion Matrix (Row=Actual, Column=Predicted):')
cm = confusion_matrix(y, calibrated_oof_preds)
print(pd.DataFrame(cm, index=labels, columns=labels))

# **Cell 6: Generate and Verify Submission CSV**

In [ ]:
calibrated_test_preds = np.argmax(final_test_prob * best_multipliers, axis=1)
# Advanced Physical Post-Processing สำหรับเคสที่ความน่าจะเป็น walk กับ stairs_up สูสีกันมาก
p_walk = (final_test_prob * best_multipliers)[:, label_map['walk']]
p_up = (final_test_prob * best_multipliers)[:, label_map['stairs_up']]
ambig_walk_up = np.abs(p_walk - p_up) < 0.12
# การขึ้นบันไดต้องใช้พลังงานส่งตัวในแนวดิ่งอย่างต่อเนื่อง (Positive Vertical Power)
stair_rule = ambig_walk_up & (X_test_feat['power_v_pos_ratio'] > 0.52) & (X_test_feat['ratio_energy_f_v'] < 1.05)
calibrated_test_preds[stair_rule] = label_map['stairs_up']
predicted_activities = [inv_label_map[p] for p in calibrated_test_preds]
submission = pd.DataFrame({
    'id': test_df['id'],
    'activity': predicted_activities
})
# ตรวจสอบความถูกต้องครบถ้วน
assert len(submission) == len(test_df), f"Row count mismatch! {len(submission)} vs {len(test_df)}"
assert submission['activity'].isnull().sum() == 0, "Submission contains null values!"
assert set(submission['activity'].unique()).issubset(set(labels)), "Submission contains invalid activity labels!"
# บันทึกตรงไปยัง /kaggle/working/submission.csv
submission.to_csv(submission_path, index=False)
print(f'\n✅ Successfully generated and saved to: {submission_path}!')
print(f'File shape: {submission.shape}')
print(f'File exists in output: {os.path.exists(submission_path)}')
print('\nPredicted Activity Distribution in Test Set:')
print(submission['activity'].value_counts())
print('\nPreview of submission.csv:')
print(submission.head(10))